# 6. Allocation of Computing Resources

A QUEENS scheduler splits a fixed core budget with two parameters: how many jobs run at the same time (`num_jobs`) and how many cores each single job gets (`num_procs`). 

This tutorial runs the same study under different allocations, shows when an allocation leads to idle cores, and fixes that by splitting up the study into multiple optimally-sized runs.

Absolute timings depend on the computer used, so every result below is read as a *ratio*. Those ratios carry over to any hardware; the runtimes themselves do not.


## The Parameters

| parameter | meaning |
|------|---------|
| `num_jobs`  | how many jobs run at the same time |
| `num_procs` | how many cores each single job may use |

**cores in use = `num_jobs × num_procs`**

The scheduler passes `num_procs` to the driver's `run(...)`; how the driver spends those cores is up to you. Which split is right depends on the solver, e.g. a serial solver wants many one-core jobs.

Cluster schedulers include a third parameter: `num_nodes` - which allocated cluster nodes per job; `num_procs` then assigns cores per node, so one job gets `num_nodes × num_procs` cores while `num_jobs` still sets how many such jobs run at once.

---

*Example:*

On a cluster whose nodes have 24 cores each: to run three jobs at the same time, each on 48 cores, set `num_jobs = 3`, `num_nodes = 2` and `num_procs = 24`. Each job then spans two nodes (2 × 24 = 48 cores), and the three jobs together occupy `num_jobs × num_nodes × num_procs = 144 cores`.


In [ ]:
import logging
import os
import subprocess
import sys
import time
from pathlib import Path

import numpy as np

from queens.distributions import Uniform
from queens.drivers._driver import Driver
from queens.global_settings import GlobalSettings
from queens.iterators import MonteCarlo
from queens.main import run_iterator
from queens.models import Simulation
from queens.parameters import Parameters
from queens.schedulers import Local
from queens.utils.io import load_result

logging.getLogger("distributed").setLevel(logging.WARNING)              # hides unnecessary output
os.environ["DASK_DISTRIBUTED__LOGGING__DISTRIBUTED"] = "warning"        # hides unnecessary output

N_POINTS = 10000000     # points per pi estimate; USER: try different values to feel the difference
SEED = 17               # master seed of the whole tutorial

## Example: Estimating $\pi$

Throw points $(x, y)$, uniformly distributed on a unit square $[0,1] \times [0,1]$. The probability that a single point lands inside (hits) the quarter circle is purely geometric - the ratio of the two areas:

$$P(x^2 + y^2 \le 1) = \frac{\text{area of quarter circle}}{\text{area of square}} = \frac{\pi/4}{1} = \frac{\pi}{4}$$

This is not an approximation: $\pi$ is contained exactly in this probability before a single point is thrown. And a probability can be measured by repetition - the
more points you throw, the closer the fraction of hits gets to $\pi/4$ (law of large numbers):

$$\hat{p}_N = \frac{\text{hits}}{N} \;\xrightarrow{N \to \infty}\; \frac{\pi}{4} \qquad \Rightarrow \qquad \pi_{\text{est}} = 4 \cdot \hat{p}_N \to \pi$$

---

*Examples:* 

With 10 throws ($N = 10$) and 8 hits in the quarter circle
$$\hat{p}_{10} = \frac{8}{10} = 0.8 \qquad \Rightarrow \qquad \pi_{\text{est}} = 4 \cdot 0.8 = 3.2$$
With 1000000 throws and 785398 hits in the quarter circle
$$\hat{p}_{10^6} = \frac{785398}{10^6} = 0.785398 \qquad \Rightarrow \qquad \pi_{\text{est}} = 4 \cdot 0.785398 = 3.141592$$
The error decays like $1/\sqrt{N}$, so a decent estimate needs millions of points.

### Step 1: One Estimate on One Core

`count_hits` counts the hits inside the quarter circle. `N_POINTS` is a *runtime* parameter, not an accuracy requirement: the higher this number the more time it takes on a single core - that gives the parallelisation below something to speed up.

In [ ]:
def count_hits(n_points, seed):
    rng = np.random.default_rng(seed)
    x = rng.random(n_points)
    y = rng.random(n_points)
    return int(np.count_nonzero(x**2 + y**2 <= 1.0))


start = time.perf_counter()
hits = count_hits(N_POINTS, SEED)
print(f"hits = {hits}")
pi_est = 4.0 * hits / N_POINTS
print(f"pi ~ {pi_est:.5f}  ({time.perf_counter() - start:.2f} sec with 1 core)")

### Step 2: One Estimate on Several Cores

To make a *single* estimate faster, split it into two halves that will carry through the rest of the tutorial:

- **`split_work`** - the cheap, serial half: divide `N_POINTS` into `num_chunks` chunks, each with its own seed so the chunks draw independent points.
- **`run_chunks`** - the expensive, parallel half: one worker process per chunk counts its hits, then the hits are combined.

`estimate_pi` is just the two halves glued together.

Don't expect the speedup to match the increase in cores: with `num_procs = 4` the ideal would be a 4x speedup, but you will measure clearly less. Every worker is a fresh Python process, and starting one - launching the interpreter, importing numpy, etc. - takes a fixed amount of time that stays the same no matter how many cores you add.

In [ ]:
WORKER = """
import sys, numpy as np
n_points, seed = int(sys.argv[1]), int(sys.argv[2])
rng = np.random.default_rng(seed)
x = rng.random(n_points)
y = rng.random(n_points)
print(int(np.count_nonzero(x ** 2 + y ** 2 <= 1.0)))
"""


def split_work(n_points, seed, num_chunks):
    """Serial half: split one estimate into `num_chunks` chunks, each with its own seed."""
    seeds = np.random.SeedSequence(seed).generate_state(num_chunks)
    counts = [n_points // num_chunks] * num_chunks
    counts[-1] += n_points - sum(counts)  # last chunk takes the remainder
    return list(zip(counts, seeds))


def run_chunks(chunks):
    """Parallel half: one worker process per chunk, combine the hits."""
    workers = [
        subprocess.Popen(
            [sys.executable, "-c", WORKER, str(n), str(s)], stdout=subprocess.PIPE, text=True
        )
        for n, s in chunks
    ]
    hits = sum(int(w.communicate()[0]) for w in workers)
    total = sum(n for n, _ in chunks)
    return 4.0 * hits / total


def estimate_pi(n_points, seed, num_procs):
    return run_chunks(split_work(n_points, seed, num_procs))


times = {}
for num_procs in (1, 4):    # USER: try different values to feel the difference
    start = time.perf_counter()
    pi_est = estimate_pi(N_POINTS, SEED, num_procs)
    times[num_procs] = time.perf_counter() - start
    print(f"pi ~ {pi_est:.5f}  ({times[num_procs]:.2f} sec with {num_procs} core(s))")

print(f"\nspeedup with 4 cores: {times[1] / times[4]:.1f}x (ideally: 4x)")

### Step 3: The same study in QUEENS

Now we hand the orchestration to QUEENS.

- **`num_samples`** - how many samples to run in *total*.
- **`num_jobs`** - how many of those samples run *at the same time*.
- **`num_procs`** - how many cores *each job* gets.

---

*Example:* 

`num_samples = 8` with `num_jobs = 4` and `num_procs = 1` means that four jobs run at a given time with one core each until eight samples are done.

---

**Reading the timings:** several outputs show up, each measuring something different:

- **`Time for CALCULATION: … s`** (QUEENS) - wall-clock time of the calculation.
- **`… it/s`** (progress bar) - throughput: samples finished per second of that calculation time.
- **`… s per sample`** (our own print) - the compute time of a single sample inside the driver.
- **`… s for the whole study`** (our own print) - **setup** (start the worker pool) **+ calculation** (the number QUEENS reports above) **+ teardown** (save results, stop the workers).

In [ ]:
class PiDriver(Driver):
    """One QUEENS job = one pi estimate, spending its `num_procs` on the workers."""

    def __init__(self, parameters, n_points):
        super().__init__(parameters=parameters)
        self.n_points = n_points

    def run(self, sample, job_id, num_procs, experiment_dir, experiment_name):
        # the sampled float in [0, 1] becomes this job's integer seed
        seed = int(self.parameters.sample_as_dict(sample)["seed"] * 1e9)
        start = time.perf_counter()
        pi_est = estimate_pi(self.n_points, seed, num_procs)
        elapsed = time.perf_counter() - start
        return {"result": np.array([pi_est]), "time": np.array([elapsed])}


parameters = Parameters(seed=Uniform(lower_bound=0.0, upper_bound=1.0))  # each job's only input


def run_study(experiment_name, driver, num_jobs, num_procs, num_samples):
    """Run one MonteCarlo study under a given (num_jobs, num_procs) allocation."""
    with GlobalSettings(experiment_name=experiment_name, output_dir="./output") as gs:
        scheduler = Local(
            experiment_name=gs.experiment_name,
            num_jobs=num_jobs,
            num_procs=num_procs,
            overwrite_existing_experiment=True,
        )
        model = Simulation(scheduler=scheduler, driver=driver)
        iterator = MonteCarlo(
            model=model,
            parameters=parameters,
            global_settings=gs,
            seed=SEED,
            num_samples=num_samples,
            result_description={"write_results": True, "plot_results": False},
        )
        run_iterator(iterator, global_settings=gs)
        return load_result(gs.result_file(".pickle"))["raw_output_data"]


def report(num_jobs, num_procs, num_samples):
    """Run one allocation and print per-sample and whole-study timings."""
    start = time.perf_counter()
    out = run_study(
        experiment_name=f"PI_EXPERIMENT_{num_jobs}_jobs_{num_procs}_procs",
        driver=PiDriver(parameters, n_points=N_POINTS),
        num_jobs=num_jobs,
        num_procs=num_procs,
        num_samples=num_samples,
    )
    study_time = time.perf_counter() - start
    estimates = np.array(out["result"]).ravel()
    times = np.array(out["time"]).ravel()
    for i, (pi_i, t_i) in enumerate(zip(estimates, times), start=1):
        print(f"Sample {i}: pi ~ {pi_i:.5f}  ({t_i:.2f} sec with {num_procs} core(s))")
    print(f"\nmean pi ~ {estimates.mean():.5f} over {len(estimates)} samples")
    print(f"mean time ~ {times.mean():.2f} sec per sample")
    print(f"{study_time:.2f} sec for the whole study (setup + calculation + teardown)\n")


for num_jobs, num_procs in [(4, 1), (1, 4)]:    # USER: try (2, 2), or raise num_samples; try different values to feel the difference
    report(num_jobs=num_jobs, num_procs=num_procs, num_samples=8)

### Step 4: Jobs with phases

Real jobs are rarely pure heavy computation - they run in *phases*. A simulation first does cheap serial preprocessing (meshing, reading input files) on **one** core, then the expensive solver runs on many, and at the end a light postprocessing step is serial again. A QUEENS job holds **one** allocation from start to finish, so `num_procs` must be sized for the hungriest phase - the solver - and during every cheap phase most of those cores sit idle.

`TwoPhaseDriver` mimics the first two phases with the two halves of `estimate_pi`:

- **preparation** (e.g. meshing): `split_work`, stretched by a `PREPARATION_SECONDS` sleep to give it a realistic weight - serial, uses one core.
- **computation** (e.g. simulation): `run_chunks` on `num_procs` cores.

Postprocessing would simply be a third phase - everything below extends to any number of phases.

In [ ]:
PREPARATION_SECONDS = 1.0   # cost of preparation
SAMPLES = 8                 # USER: try different values to feel the difference
PROCS = 4                   # USER: try different values to feel the difference
JOBS = 1                    # USER: try different values to feel the difference


class TwoPhaseDriver(Driver):
    """One job = serial preparation (1 core) + parallel computation (`num_procs` cores)."""

    def __init__(self, parameters, n_points):
        super().__init__(parameters=parameters)
        self.n_points = n_points

    def run(self, sample, job_id, num_procs, experiment_dir, experiment_name):
        seed = int(self.parameters.sample_as_dict(sample)["seed"] * 1e9)
        t0 = time.perf_counter()
        time.sleep(PREPARATION_SECONDS)     # serial preparation (1 core)
        chunks = split_work(self.n_points, seed, num_procs)
        t1 = time.perf_counter()
        pi = run_chunks(chunks)             # parallel simulation, (PROCS cores)
        t2 = time.perf_counter()
        return {
            "result": np.array([pi]),
            "preparation": np.array([t1 - t0]),
            "computation": np.array([t2 - t1]),
        }


start = time.perf_counter()
out = run_study(
    experiment_name="PI_EXPERIMENT_1_driver",
    driver=TwoPhaseDriver(parameters, N_POINTS),
    num_jobs=JOBS,
    num_procs=PROCS,
    num_samples=SAMPLES,
)
one_driver_time = time.perf_counter() - start  # compared against the two-driver version below

estimates = np.array(out["result"]).ravel()
preparation = np.array(out["preparation"]).ravel()
computation = np.array(out["computation"]).ravel()
times = preparation + computation

for i, (pi_i, p_i, c_i) in enumerate(zip(estimates, preparation, computation), start=1):
    print(f"Sample {i}: pi ~ {pi_i:.5f}  ({p_i:.2f} sec preparation + {c_i:.2f} sec computation with {PROCS} core(s))")

idle = (PROCS - 1) * preparation.sum()  # core-seconds idling during preparation
allocated = PROCS * times.sum()         # core-seconds the allocation holds in total

print(f"\nmean pi ~ {estimates.mean():.5f} over {len(estimates)} samples")
print(f"mean time ~ {preparation.mean():.2f} sec preparation + {computation.mean():.2f} sec computation per sample\n")
print(f"{one_driver_time:.2f} sec for the whole study (setup + calculation + teardown)\n")
print(f"Total allocated time ~ {allocated:.2f} sec (total compute time)")
print(f"Total idle time ~ {idle:.2f} sec")
print(f"Unused compute time (idle/allocated) ~ {idle / allocated:.0%}")

#### The fix: one driver per phase

Nothing forces both phases to share one allocation. Split the driver into two, and give each its own study with its own allocation:

1. **`PreparationDriver`** it builds the chunk plan (`split_work`) and writes it to the disk.
2. **`ComputationDriver`** it reads the plan and runs it (`run_chunks`).

Both studies use the same iterator seed, so the computation study sees the same samples - and the same `job_id`s - as the preparation study; that is how each computation job finds its plan file.

In [ ]:
PLAN_DIR = Path("./output/pi_plans")  # PreparationDriver writes plans here, ComputationDriver reads them

PREPARATION_SECONDS = 1.0   # cost of preparation
PREPARATION_JOBS = 4        # USER: try different values to feel the difference
PREPARATION_PROCS = 1       # USER: try different values to feel the difference

COMPUTATION_JOBS = 1        # USER: try different values to feel the difference
COMPUTATION_PROCS = 4       # USER: try different values to feel the difference

SAMPLES = 8                 # USER: try different values to feel the difference


class PreparationDriver(Driver):
    """Serial phase only: build the chunk plan, write it to disk."""

    def __init__(self, parameters, n_points):
        super().__init__(parameters=parameters)
        self.n_points = n_points

    def run(self, sample, job_id, num_procs, experiment_dir, experiment_name):
        seed = int(self.parameters.sample_as_dict(sample)["seed"] * 1e9)
        start = time.perf_counter()
        time.sleep(PREPARATION_SECONDS)
        chunks = split_work(self.n_points, seed, COMPUTATION_PROCS)  # plan for the computation study
        np.save(PLAN_DIR / f"plan_{job_id}.npy", np.array(chunks))
        elapsed = time.perf_counter() - start
        return {"result": np.array([float(len(chunks))]), "time": np.array([elapsed])}


class ComputationDriver(Driver):
    """Parallel phase only: read the plan back, run the chunks on `num_procs` cores."""

    def run(self, sample, job_id, num_procs, experiment_dir, experiment_name):
        start = time.perf_counter()
        chunks = np.load(PLAN_DIR / f"plan_{job_id}.npy")
        pi = run_chunks([(int(n), int(s)) for n, s in chunks])
        elapsed = time.perf_counter() - start
        return {"result": np.array([pi]), "time": np.array([elapsed])}


PLAN_DIR.mkdir(parents=True, exist_ok=True)

start = time.perf_counter()
out_preparation = run_study(
    experiment_name="PI_EXPERIMENT_PREPARATION",
    driver=PreparationDriver(parameters, N_POINTS),
    num_jobs=PREPARATION_JOBS,
    num_procs=PREPARATION_PROCS,
    num_samples=SAMPLES,
)
preparation_time = time.perf_counter() - start

start = time.perf_counter()
out_computation = run_study(
    experiment_name="PI_EXPERIMENT_COMPUTATION",
    driver=ComputationDriver(parameters),
    num_jobs=COMPUTATION_JOBS,
    num_procs=COMPUTATION_PROCS,
    num_samples=SAMPLES,
)
computation_time = time.perf_counter() - start
two_driver_time = preparation_time + computation_time

print(f"Preparation study ({PREPARATION_JOBS} job(s) x {PREPARATION_PROCS} core(s)):")
print("-" * 50)
chunk_counts = np.array(out_preparation["result"]).ravel()
times = np.array(out_preparation["time"]).ravel()
# for i, (n_i, t_i) in enumerate(zip(chunk_counts, times), start=1): # USER: uncomment if needed
#     print(f"Sample {i}: Split into {int(n_i)} chunks  ({t_i:.2f} sec with {PREPARATION_PROCS} core(s))")
print(f"mean time ~ {times.mean():.2f} sec per sample")
print(f"{preparation_time:.2f} sec for the whole study (setup + calculation + teardown)\n")

print(f"Computation study ({COMPUTATION_JOBS} job(s) x {COMPUTATION_PROCS} core(s)):")
print("-" * 50)
estimates = np.array(out_computation["result"]).ravel()
times = np.array(out_computation["time"]).ravel()
for i, (pi_i, t_i) in enumerate(zip(estimates, times), start=1):
    print(f"Sample {i}: pi ~ {pi_i:.5f}  ({t_i:.2f} sec with {COMPUTATION_PROCS} core(s))")
print(f"\nmean pi ~ {estimates.mean():.5f} over {len(estimates)} samples")
print(f"mean time ~ {times.mean():.2f} sec per sample\n")
print(f"{computation_time:.2f} sec for the whole study (setup + calculation + teardown)\n")

print(f"One driver ('TwoPhaseDriver') ~ {one_driver_time:.2f} sec (see the previous output)")
print(f"Two drivers ('PreparationDriver' = {preparation_time:.2f} sec + 'ComputationDriver' = {computation_time:.2f} sec) ~ {two_driver_time:.2f} sec")
print(f"Splitting up the study into multiple drivers is {one_driver_time / two_driver_time:.1f}x faster!")

## Lessons Learned

- The computing power in use is the product of the two scheduler parameters: **cores in use = `num_jobs × num_procs`**. The product is bounded by the available cores of the computer; the split between the two factors is up to the user.
- An allocation of computing power is **fixed** from start to finish. If the job runs in phases, `num_procs` must be sized for the most computationally demanding phase - and during every cheaper phase, the extra cores sit idle.
- Splitting the study into one driver per phase gives control to split the computing power optimally - the idle time turns into throughput (compare the one-driver and two-driver timings above).

The same logic applies on a cluster, where `num_nodes` joins the split as a third factor (see *The Parameters* above).